# Denoising pile driving

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
from adaptfilt import lms, nlms, rls, nlmsru, mswe, ap
import emd
import matplotlib.pyplot as plt
import numpy as np
import pywt
from scipy.signal import correlate, correlation_lags, find_peaks, welch, get_window, medfilt, convolve
from tritonoa.data.reader import read_hdf5, read_inventory
from tritonoa.data.signal import taper
from tritonoa.data.time import TIME_PRECISION

from vineyard.config import SENSORS, get_path
from vineyard.filters import ukf_reference_denoiser, spectral_subtraction, cepstral_subtraction, plot_signal_cwt

FIGWIDTH = 14

In [ ]:
def load_data(
    sensor: str,
    frequencies: list[float] = [20.0, 25.0],
    channels: int = 3,
    time_start: np.datetime64 = np.datetime64("2023-12-01 22:21:00", TIME_PRECISION),
    time_end: np.datetime64 = np.datetime64("2023-12-01 22:26:00", TIME_PRECISION),
):
    inventory = get_path(f"{sensor}_inventory")
    ds = read_inventory(
        inventory,
        time_start=time_start,
        time_end=time_end,
        channels=channels,
    ).decimate(20).filter(filt_type="bandpass", freq=frequencies)
    ds.data /= 1e6
    return ds.time_vector, ds.data[0], ds.stats.sampling_rate


sensor = "vla1"
# frequencies = [15.0, 50.0]
frequencies = [19.0, 25.0]
channels = 3
time_start = np.datetime64("2023-12-01 22:23:00", TIME_PRECISION)
time_end = np.datetime64("2023-12-01 22:24:00", TIME_PRECISION)

# time_start = np.datetime64("2023-12-01 21:51:22", TIME_PRECISION)
# time_end = np.datetime64("2023-12-01 21:51:32", TIME_PRECISION)

t, x, fs = load_data(
    sensor,
    frequencies=frequencies,
    channels=channels, 
    time_start=time_start,
    time_end=time_end,
)
# x = medfilt(x, kernel_size=3)

time_start_s = np.datetime64("2023-12-01 22:25:20", TIME_PRECISION)
time_end_s = np.datetime64("2023-12-01 22:25:21.8", TIME_PRECISION)
ts, xs, _ = load_data(
    sensor,
    # frequencies=[15.0, 50.0],
    frequencies=frequencies,
    channels=channels,
    time_start=time_start_s,
    time_end=time_end_s,
)
xs *= taper(len(xs), max_percentage=0.05)

time_start_w = np.datetime64("2023-12-01 22:25:46", TIME_PRECISION)
time_end_w = np.datetime64("2023-12-01 22:25:47", TIME_PRECISION)
tw, xw, _ = load_data(
    sensor,
    # frequencies=[19.0, 25.0],
    frequencies=frequencies,
    channels=channels,
    time_start=time_start_w,
    time_end=time_end_w,
)
xw *= taper(len(xw), max_percentage=0.05)

In [ ]:
plt.figure(figsize=(FIGWIDTH, 3))
plt.plot(t, x, label="Original Signal")
plt.grid()
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

plt.figure(figsize=(FIGWIDTH, 3))
plt.plot(ts, xs, label="Original Signal")
plt.grid()
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Strike Signal")
plt.tight_layout()
plt.show()

plt.figure(figsize=(FIGWIDTH, 3))
plt.plot(tw, xw, label="Original Signal")
plt.grid()
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Whale Signal")
plt.tight_layout()
plt.show()

In [ ]:
sig1 = xs
sig2 = xs
xcorr = correlate(sig1, sig2, mode="full")
lags = np.arange(-len(sig1) + 1, len(sig2))
plt.figure(figsize=(FIGWIDTH, 3))
plt.plot(lags, xcorr)
plt.grid()
plt.xlabel("Lag")
plt.ylabel("Cross-correlation")
plt.title("Autocorrelation of Strike Signal")
plt.tight_layout()
plt.show()

sig1 = xw
sig2 = xw
xcorr = correlate(sig1, sig2, mode="full")
lags = np.arange(-len(sig1) + 1, len(sig2))
plt.figure(figsize=(FIGWIDTH, 3))
plt.plot(lags, xcorr)
plt.grid()
plt.xlabel("Lag")
plt.ylabel("Cross-correlation")
plt.title("Autocorrelation of Whale Signal")
plt.tight_layout()
plt.show()

In [ ]:
xcorr = correlate(x, xs, mode="same")
# xcorr /= np.sqrt(np.sum(xs**2) * np.sum(x**2))
xcorr /= np.max(np.abs(xcorr))  # Normalize the cross-correlation
print(xcorr.min(), xcorr.max())

lags = correlation_lags(len(x), len(xs), mode="same")

peaks = find_peaks(xcorr, height=0.4, distance=int(1.2 * fs))[0]
peak_amps = xcorr[peaks]


plt.figure(figsize=(FIGWIDTH, 3))
plt.plot(t, x / np.max(np.abs(x)))
plt.plot(t, xcorr, label="Cross-correlation between signal & strike")
plt.plot(t[peaks], peak_amps, "ro", label="Peaks")
# [plt.axvline(t[pk], color="r") for pk in peaks]
# plt.axvline(t[np.argmax(xcorr)], color="r")

plt.grid()
plt.xlabel("Lag")
plt.ylabel("Cross-correlation")
plt.title("Cross-correlation between Full Signal and Strike")
plt.tight_layout()
plt.show()

# xcorr = correlate(x, xw, mode="same")
# # xcorrnorm = np.sqrt(np.sum(xe**2) * np.sum(x**2))
# xcorr /= np.max(np.abs(xcorr))  # Normalize the cross-correlation

# lags = correlation_lags(len(x), len(xw), mode="same")

# peaks = find_peaks(xcorr, height=0.5, distance=int(1.0 * fs))[0]

# plt.figure(figsize=(FIGWIDTH, 3))
# plt.plot(t, x / np.max(np.abs(x)))
# plt.plot(t, xcorr, label="Cross-correlation between signal & whale")
# [plt.axvline(t[pk], color="r") for pk in peaks]
# plt.grid()
# plt.xlabel("Lag")
# plt.ylabel("Cross-correlation")
# plt.title("Cross-correlation between Full Signal and Whale")
# plt.tight_layout()
# plt.show()

# plt.figure(figsize=(10, 3))
# plt.plot(lags, xcorr)
# plt.grid()
# plt.xlabel("Lag")
# plt.ylabel("Cross-correlation")
# plt.title("Cross-correlation between Full Signal and Excerpt")
# plt.tight_layout()
# plt.show()

In [ ]:
np.diff(lags[peaks])  # Check the lag differences between peaks

In [ ]:
# whale_call_inds = [3]
# whale_call_inds = [3, 8, 9, 14, 19, 20, 22]
whale_call_inds = [3, 4, 8, 9, 14, 20, 25, 30, 35, 40, 41, 46, 51, 52, 54]

plt.figure(figsize=(FIGWIDTH, 4))
for pk, ind in enumerate(peaks):
    start = int(ind - 1.0 * fs)
    end = int(ind + 1.0 * fs)
    segment = x[start:end]
    f, psd = welch(segment, fs=fs, nperseg=8192, nfft=2 ** 16)
    # f, psd = welch(segment, fs=fs, nperseg=64, noverlap=60, nfft=2 ** 15)
    if pk in whale_call_inds:
        plt.plot(f, psd, "r--")
    else:
        plt.plot(f, psd, "k-")
plt.xlim([10, 50])
# plt.legend()
plt.show()

In [ ]:
start_buffer = 0.9
end_buffer = 0.7

fig = plt.figure(figsize=(FIGWIDTH, 4))
tseg = np.arange(-start_buffer, end_buffer, 1 / fs)[:-1]
for pk, ind in enumerate(peaks):
    start = int(ind - start_buffer * fs)
    end = int(ind + end_buffer * fs)
    segment = x[start:end]
    if pk in whale_call_inds:
        style = "r-"
    else:
        style = "k-"
    plt.plot(tseg, segment / np.max(np.abs(segment)) + pk, style)
    plt.gca().invert_yaxis()
plt.show()

fig = plt.figure(figsize=(FIGWIDTH, 4))
tseg = np.arange(-start_buffer, end_buffer, 1 / fs)[:-1]
for pk, ind in enumerate(peaks):
    start = int(ind - start_buffer * fs)
    end = int(ind + end_buffer * fs)
    segment = x[start:end]
    if pk in whale_call_inds:
        style = "r-"
        continue
    else:
        style = "k-"
    plt.plot(tseg, segment / np.max(np.abs(segment)), style)
plt.show()

In [ ]:
segments_no_w = []
for pk, ind in enumerate(peaks):
    if pk in whale_call_inds:
        continue
    start = int(ind - start_buffer * fs)
    end = int(ind + end_buffer * fs)
    segment = x[start:end]
    segments_no_w.append(segment)

segments_no_w = np.array(segments_no_w)
segment_avg_no_w = np.mean(segments_no_w, axis=0)

tap = taper(len(segment), max_percentage=0.05)
# window = "tukey"
# taper_len = len(segment_avg_no_w) // 2
# taper = get_window(window, taper_len)
# print(len(taper), len(tseg))
# taper = np.concatenate((taper[:taper_len], np.zeros(len(tseg) - len(taper)), taper[:taper_len]))
segment_avg_no_w *= tap

segments_w = []
for pk, ind in enumerate(peaks):
    start = int(ind - start_buffer * fs)
    end = int(ind + end_buffer * fs)
    segment = x[start:end]
    segments_w.append(segment)

segments_w = np.array(segments_w)
segment_avg_w = np.mean(segments_w, axis=0)

# window = "tukey"
# taper = get_window(window, len(segment_avg_w))
segment_avg_w *= tap



plt.figure(figsize=(FIGWIDTH, 4))
plt.plot(tseg, segment_avg_w, label="w/ Whale Calls")
plt.plot(tseg, segment_avg_no_w, label="w/o Whale Calls")
plt.plot(tseg, tap * np.max(np.abs(segment_avg_w)))
plt.xlabel("Lag (s)")
plt.legend()
plt.title("Filter Template")
plt.show()

In [ ]:
template = segment_avg_no_w
ir = np.zeros(len(x))
ir[peaks] = peak_amps

ir = convolve(ir, template, mode="same")

residual = x - ir

fig, axes = plt.subplots(nrows=3)
ax = axes[0]
ax.plot(x)
ax = axes[1]
ax.plot(ir)
ax = axes[2]
ax.plot(residual)
plt.show()

# synth_record = convolve()

In [ ]:
xcorr = correlate(residual, xw, mode="same")
# xcorr /= np.sqrt(np.sum(xs**2) * np.sum(x**2))
xcorr /= np.max(np.abs(xcorr))  # Normalize the cross-correlation

plt.figure(figsize=(FIGWIDTH, 3))
plt.plot(t, x / np.max(np.abs(x)))
plt.plot(t, xcorr, label="Cross-correlation between signal & strike")

## Filter data

In [ ]:
from tqdm.notebook import tqdm

num_taps: int = 32
alpha: float = 0.01
ffactor: float = 0.99999
K: int = 16

scale = np.max(np.abs(x))
x_scaled = np.copy(x) / scale
template = segment_avg_w / scale
# template = 

x_filtered = x_scaled.copy()
y_arr = np.zeros_like(x)
w_init = None
w_it = np.empty((len(t), num_taps))
w_it[:] = np.nan  # Initialize with NaN for unused taps

# for pk, ind in enumerate([peaks[14]]):
for pk, ind in tqdm(enumerate(peaks), total=len(peaks)):
    start = int(ind - start_buffer * fs)
    end = int(ind + end_buffer * fs)
    segment = x_scaled[start:end].copy()
    if segment.shape[0] != len(template):
        continue

    # e = segment - template
    # e = spectral_subtraction(segment, template, beta=1e-5)
    # e = cepstral_subtraction(tap * segment, tap * template, len(segment))
    # y = np.zeros_like(segment)
    w = np.zeros((len(segment), num_taps))
    
    # y, e, w = lms(segment, template, num_taps, alpha, leak=0.0, initCoeffs=w_init, N=None, returnCoeffs=True)
    # y, e, w = nlms(segment, template, num_taps, alpha, initCoeffs=w_init, returnCoeffs=True)
    # y, e, w = rls(segment, template, num_taps, ffactor, initCoeffs=w_init, N=None, returnCoeffs=True)
    # y, e, w = rls(segment, e_, num_taps, ffactor, initCoeffs=w_init, N=None, returnCoeffs=True)

    # print(segment.min(), segment.max())
    # print(template.min(), template.max())
    
    # y, e, w = ap(
    #     segment,
    #     template,
    #     num_taps,
    #     alpha,
    #     K=K,
    #     leak=0.0,
    #     initCoeffs=w_init,
    #     N=None,
    #     returnCoeffs=True,
    # )
    e, y = ukf_reference_denoiser(
        segment,
        template,
        process_noise=0.01,
        measurement_noise=0.1,
        dt=1.0/fs,  # Convert to time step
    )

    # w_init = np.mean(w[-10:, :], axis=0)  # Running average of the last 10 taps

    y_arr[start:end] = y
    x_filtered[start:end] = e
    w_it[start:end, :] = w

    # w_init = w[-1, :]
    # y_arr[start + num_taps - 1:end] = y
    # x_filtered[start + num_taps - 1:end] = e
    # w_it[start + num_taps - 1:end, :] = w

    # y_arr[start + num_taps + K - 1:end] = y
    # x_filtered[start + num_taps + K - 1:end] = e
    # w_it[start + num_taps + K - 1:end, :] = w

    # x_filtered[start:end] = e
    # break

# e /= np.max(np.abs(e))

# plt.figure(figsize=(FIGWIDTH, 3))
# plt.plot(segment, label="Original Signal")
# plt.plot(segment - segment_avg / scale, label="Original Signal")
# # plt.plot(e / np.max(np.abs(e)), label="Filtered Signal")
# plt.grid()
# plt.xlabel("Time (s)")
# plt.ylabel("Amplitude")
# plt.show()

In [ ]:
fig, axs = plt.subplots(nrows=3, figsize=(FIGWIDTH, 8), sharex=True)
ax = axs[0]
# fig, axs = plt.subplots(nrows=1, figsize=(FIGWIDTH, 8), sharex=True)
# ax = axs
ax.plot(x_scaled, label="Original Signal")
ax.plot(x_filtered, label="Filtered Signal")
ax.set_xlim([0, len(x)])
ax.set_xlabel("Sample")
ax.set_ylabel("Amplitude")
ax.set_title("LMS Error")
ax.grid()
ax.legend()

ax = axs[1]
ax.plot(x_scaled, label="Original Signal")
ax.plot(y_arr, label="Filtered Signal")
ax.set_xlim([0, len(x)])
ax.set_xlabel("Sample")
ax.set_ylabel("Amplitude")
ax.set_title("LMS Estimated Signal")
ax.grid()
ax.legend()

ax = axs[2]
# for i, w in enumerate(w_it):
    # ax.plot(i * len(w) + np.arange(0, len(w)), w, "k")
plt.plot(w_it, "k")
ax.set_xlabel("Iteration")
ax.set_ylabel("Filter Coefficients")
ax.set_title("LMS Filter Coefficients")
ax.set_xlim([0, len(x)])
ax.grid()
plt.tight_layout()
plt.show()


In [ ]:
height = 0.25

xcorr_filt = correlate(x_filtered, xs / scale, mode="same")
xcorr_filt /= np.max(np.abs(xcorr_filt))  # Normalize the cross-correlation
# xcorr_filt /= np.sqrt(np.sum(x_filtered**2) * np.sum(xs**2))  # Normalize by energy
# xcorr_filt /= np.max(np.abs(xcorr_filt))  # Normalize the cross-correlation
lags_filt = correlation_lags(len(x_filtered), len(xw), mode="same")

peaks_filt = find_peaks(xcorr_filt, height=height, distance=int(8.0 * fs))[0]

fig, axes = plt.subplots(figsize=(FIGWIDTH, 8), nrows=2, sharex=True)
plt.subplot(211)
plt.plot(t, x_scaled, label="Original Signal")
plt.plot(t, xcorr_filt, label="Cross-correlation")
plt.plot(t[peaks_filt], xcorr_filt[peaks_filt], "ro", label="Peaks")
# [plt.axvline(t[pk], color="r") for pk in peaks_filt]
plt.grid()
plt.xlabel("Lag")
plt.ylabel("Cross-correlation")
plt.title("Cross-correlation between Filtered Signal and Whale")
plt.legend()
plt.tight_layout()


xcorr_filt_w = correlate(x, xw / scale, mode="same")
xcorr_filt_w /= np.max(np.abs(xcorr_filt_w))  # Normalize the cross-correlation
lags_filt = correlation_lags(len(x_filtered), len(xw), mode="same")

peaks_filt = find_peaks(xcorr_filt_w, height=height, distance=int(1.0 * fs))[0]

# plt.figure(figsize=(FIGWIDTH, 3))
plt.subplot(212)
plt.plot(t, x / np.max(np.abs(x)), label="Original Signal")
plt.plot(t, xcorr_filt_w, label="Cross-correlation")
[plt.axvline(t[pk], color="r") for pk in peaks_filt]
plt.grid()
plt.xlabel("Lag")
plt.ylabel("Cross-correlation")
plt.title("Cross-correlation between Original Signal and Whale")
plt.legend()
plt.tight_layout()
plt.show()

## EMD

In [ ]:
ix0 = 0
ix1 = int(len(x_filtered) / 2)

# print(emd.sift.get_config())
# imf = emd.sift.sift(x_filtered)
# emd.plotting.plot_imfs(imf, sample_rate=fs)
# plt.title("Original Signal IMFs")

imf = emd.sift.ensemble_sift(x_filtered, nensembles=8)
emd.plotting.plot_imfs(imf, sample_rate=fs)
plt.title("Original Signal IMFs (Ensemble Sift)")

# imf = emd.sift.iterated_mask_sift(x_filtered)
# emd.plotting.plot_imfs(imf, sample_rate=fs)
# plt.title("Original Signal IMFs (Masked Sift)")

In [ ]:
obj = xcorr_filt.copy()
# obj[obj < 0.0] = 0.0

# for i in range(100):
#     n = 0.01 * np.random.uniform(-1.0, 1.0, size=obj.shape)
#     obj += obj + n
    # obj[obj > 1.0] = 1.0

# obj = 2 - obj


# obj = obj ** (1 / 2)


plt.figure(figsize=(FIGWIDTH, 3))
plt.plot(obj)
plt.axhline(height, color="r", linestyle="--", label="Threshold")
plt.show()

In [ ]:
from vineyard.signal import complex_cepstrum, inverse_complex_cepstrum, real_cepstrum

sig = np.hstack([xs, xs])

nfft = 2 ** 10
print(nfft)
# sig = xs
print(sig.shape)

ccep, _ = complex_cepstrum(sig,  nfft)
rcep = real_cepstrum(sig, nfft)
icep = inverse_complex_cepstrum(ccep, nfft / 2)
print(ccep.shape)
print(rcep.shape)
print(icep.shape)

plt.figure()
plt.plot(ccep)
plt.plot(rcep)
plt.show()

plt.figure()
plt.plot(sig)
plt.plot(icep)
plt.show()

## Wavelets

In [ ]:
from pywt import frequency2scale

# wavelet = "cmor25.0-5.0"
wavelet = ""
scale = frequency2scale(wavelet, 35.0 / fs)
scales = np.linspace(1, scale, 100)

fig = plot_signal_cwt(xs, wavelet=wavelet, scales=scales)
plt.show()

fig = plot_signal_cwt(xw, wavelet=wavelet, scales=scales)
plt.show()

## Synthetic record

In [ ]:
# time_start = np.datetime64("2023-12-01 22:24:00", TIME_PRECISION)
# time_end = np.datetime64("2023-12-01 22:26:00", TIME_PRECISION)

# ds_synth = read_hdf5("data/acoustic/vla1_synthetic_strikes.hdf5")
# ds_synth.trim(starttime=time_start, endtime=time_end)
# xsynth = ds_synth.data[0]
xsynth = ir.copy()

xsynth /= np.max(np.abs(xsynth))

x_scaled = np.copy(x) / np.max(np.abs(x))

# print(fs, ds_synth.stats.sampling_rate)
# print(x.shape, xsynth.shape)

plt.figure(figsize=(FIGWIDTH, 3))
plt.plot(t, x_scaled, label="Original Signal")
plt.plot(t, xsynth, label="Synthetic Strike")
plt.grid()
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Synthetic Strike vs Original Signal")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
alpha: float = 0.1
num_taps: int = 16
ffactor: float = 0.999999999
# K: int = 16
# y, e, w = nlms(x_scaled, xsynth, num_taps, alpha, leak=0.0, returnCoeffs=True)
y, e, w = rls(x_scaled, xsynth, num_taps, ffactor, returnCoeffs=True)

# x_filtered = e / np.max(np.abs(e))
# y /= np.max(np.abs(y))

In [ ]:
fig, axs = plt.subplots(nrows=3, figsize=(FIGWIDTH, 8), sharex=True)
ax = axs[0]
# fig, axs = plt.subplots(nrows=1, figsize=(FIGWIDTH, 8), sharex=True)
# ax = axs
ax.plot(x_scaled, label="Original Signal")
ax.plot(e, label="Filtered Signal")
ax.set_xlim([0, len(x)])
ax.set_xlabel("Sample")
ax.set_ylabel("Amplitude")
ax.set_title("LMS Error")
ax.grid()
ax.legend()

ax = axs[1]
ax.plot(x_scaled, label="Original Signal")
ax.plot(y, label="Filtered Signal")
ax.set_xlim([0, len(x)])
ax.set_xlabel("Sample")
ax.set_ylabel("Amplitude")
ax.set_title("LMS Estimated Signal")
ax.grid()
ax.legend()

ax = axs[2]
# for i, w in enumerate(w_it):
    # ax.plot(i * len(w) + np.arange(0, len(w)), w, "k")
plt.plot(w, "k")
ax.set_xlabel("Iteration")
ax.set_ylabel("Filter Coefficients")
ax.set_title("LMS Filter Coefficients")
ax.set_xlim([0, len(x)])
ax.grid()
plt.tight_layout()
plt.show()

## Estimator-Correlator

In [ ]:
xs = xs.reshape(1, -1) / np.max(np.abs(xs))
xw = np.random.normal(0, 1e-3, size=xs.shape)  # Small noise to avoid singularity
Xs = np.fft.fft(xs).reshape(1, -1)
Xw = np.fft.fft(xw).reshape(1, -1)


# Ks = np.conj(Xs.T) @ Xs  # Covariance matrix of the signal
Ks = np.conj(xs.T) @ xs
Kw = np.conj(xw.T) @ xw  # Covariance matrix of the noise

Ks_inv = np.linalg.pinv(Ks)  # Pseudo-inverse of the covariance matrix
Kw_inv = np.linalg.pinv(Kw)

In [ ]:
plt.figure()
plt.imshow(Kw, cmap="seismic")
plt.colorbar()
plt.show()

In [ ]:
S_hat = Ks @ np.linalg.pinv(Ks + Kw) @ Xs.T
F = Xs @ np.linalg.pinv(Kw) @ S_hat

In [ ]:
F